In [69]:
import requests, random, warnings, math, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print(' Imports loaded')

 Imports loaded


In [70]:
API_KEY      = 'ok_ad14469097b36b0d1653a7d857748058'
BASE_URL     = 'https://amazon-scraper-api.omkar.cloud'
COUNTRY_CODE = 'IN'
HEADERS      = {'API-Key': API_KEY}
MODEL_PATH   = r'C:\Users\tarun\OneDrive\Desktop\Opsell\LQS_Project\models\lqs_model.pkl'

TARGET_ASIN      = 'B0CHX1W1XY'         # your product's ASIN
COMPETITOR_QUERY = 'laptop under 50000' # search query to find competitors
TOP_N_COMP       = 5

In [71]:
model = joblib.load(MODEL_PATH)
EXPECTED_COLS = list(model.feature_names_in_)
print(f'Model loaded  |  {len(EXPECTED_COLS)} features')

Model loaded  |  23 features


In [72]:
CATEGORY_KEYWORDS = {
    'laptop'   : ['laptop','notebook','chromebook','macbook','vivobook','ideapad',
                  'inspiron','pavilion','aspire','zenbook','thinkpad','elitebook',
                  'probook','gram','swift','ryzen','core i3','core i5','core i7'],
    'phone'    : ['phone','smartphone','iphone','galaxy','redmi','poco','oneplus',
                  'realme','vivo','oppo','pixel','mobile'],
    'headphone': ['headphone','earphone','earbuds','tws','airpods','headset','neckband'],
    'tv'       : ['tv','television','smart tv','qled','oled','led tv'],
    'tablet'   : ['tablet','ipad','fire hd'],
}

def infer_category(query):
    q = query.lower()
    for kws in CATEGORY_KEYWORDS.values():
        if any(kw in q for kw in kws):
            return kws
    return []

def matches_category(product, kws):
    if not kws: return True
    return any(kw in str(product.get('title','')).lower() for kw in kws)

In [73]:
# ════════════════════════════════════════════════════════════════════
# CELL 5 — Hybrid LQS 
# ════════════════════════════════════════════════════════════════════

BRAND_VOCAB = {
    'hp','dell','lenovo','asus','acer','apple','samsung','msi','lg','sony',
    'xiaomi','realme','oneplus','redmi','infinix','boat','jbl','sennheiser',
    'bose','noise','mi','iphone','macbook','thomson','toshiba','huawei',
}

def extract_features(p):
    title  = str(p.get('title','') or '')
    price  = float(p.get('price', 0) or 0)
    mrp    = float(p.get('original_price', 0) or 0)
    rating = float(p.get('rating', 0) or 0)
    rev    = float(p.get('reviews', 0) or 0)
    disc   = round((mrp - price) / mrp * 100, 2) if mrp > 0 else 0

    title_lower    = title.lower()
    brand_present  = 1 if any(b in title_lower.split() for b in BRAND_VOCAB) else 0
    sentence_case  = 1 if title and title[0].isupper() else 0
    is_established = (rating >= 4.0 and rev >= 100)

    bullet_count           = 5   if is_established else 2
    description_word_count = 200 if is_established else 40
    image_count            = 7   if is_established else 3
    has_aplus              = 1   if (brand_present and rev >= 100) else 0
    has_video              = 1   if (brand_present and rev >= 500) else 0
    bsr_proxy              = max(1, int(50000 / (rev + 10)))

    return {
        'title_length': len(title), 'title_word_count': len(title.split()),
        'title_brand_present': brand_present, 'title_sentence_case': sentence_case,
        'title_spellcheck_pass': 1, 'bullet_count': bullet_count,
        'bullets_spell_pass': 1, 'bullets_caps_pass': 1,
        'final_bullet_flag': 1 if bullet_count >= 4 else 0,
        'description_word_count': description_word_count,
        'final_desc_flag': 1 if description_word_count >= 100 else 0,
        'price': price, 'mrp': mrp, 'discount_pct': disc,
        'star_rating': rating, 'review_count': rev,
        'log_review_count': np.log1p(rev),
        'best_seller_rank': bsr_proxy, 'image_count': image_count,
        'video_count': 1 if has_video else 0, 'has_video': has_video,
        'has_aplus': has_aplus, 'aplus_check_pass': has_aplus,
    }

def heuristic_lqs(p, f):
    score = 0
    score += min(f['star_rating'] / 5.0, 1.0) * 30
    score += min(np.log1p(f['review_count']) / np.log1p(5000), 1.0) * 30
    score += 10 if f['title_brand_present'] else 0
    score += 10 if f['discount_pct'] > 10 else (5 if f['discount_pct'] > 0 else 0)
    if 80 <= f['title_length'] <= 200: score += 10
    elif f['title_length'] >= 50:      score += 5
    score += 5 if f['image_count'] >= 1 else 0
    score += 5 if (f['star_rating'] >= 4.0 and f['review_count'] >= 100) else 0
    return score

def predict_lqs(p):
    f = extract_features(p)
    X = pd.DataFrame([{c: f.get(c, 0) for c in EXPECTED_COLS}])[EXPECTED_COLS]
    model_score = float(np.clip(model.predict(X)[0], 65, 95))
    h_score = 65 + (heuristic_lqs(p, f) / 100) * 30
    final = float(np.clip(0.4 * model_score + 0.6 * h_score, 65, 95))
    grade = 'A' if final >= 88 else ('B' if final >= 75 else 'C')
    return round(final, 1), grade, f

print('✓ Hybrid LQS loaded')

✓ Hybrid LQS loaded


In [74]:
def estimate_ctr(p, f):
    s = (min(f['star_rating']/5.0, 1.0) * 30
         + min(math.log1p(f['review_count'])/math.log1p(5000), 1.0) * 25
         + (10 if f['discount_pct'] > 10 else 0)
         + (10 if f['image_count'] >= 1 else 0)
         + (10 if p.get('is_prime') else 0))
    return round(min(s/100, 1.0), 3)

def estimate_cvr(p, f):
    s = (min(f['star_rating']/5.0, 1.0) * 35
         + min(math.log1p(f['review_count'])/math.log1p(5000), 1.0) * 30
         + (15 if p.get('is_prime') else 0)
         + (10 if f['discount_pct'] > 10 else 0))
    return round(min(s/100, 1.0), 3)

def score_product(p):
    lqs, grade, f = predict_lqs(p)
    asp = f['price'] if f['price'] > 0 else 1.0
    ctr = estimate_ctr(p, f)
    cvr = estimate_cvr(p, f)
    return {
        'title': str(p.get('title',''))[:65],
        'asin' : p.get('asin', 'N/A'),
        'price': asp, 'discount': f['discount_pct'],
        'rating': f['star_rating'], 'reviews': f['review_count'],
        'lqs': lqs, 'grade': grade,
        'ctr': ctr, 'cvr': cvr,
        'rpi': round(asp * ctr * cvr, 2),
    }

In [75]:
def fetch_page(query, page=1):
    try:
        r = requests.get(f'{BASE_URL}/amazon/search',
            params={'query': query, 'country_code': COUNTRY_CODE, 'page': page},
            headers=HEADERS, timeout=60)
        return r.json().get('results', [])
    except Exception as e:
        print(f'  page {page} error: {e}')
        return []

def fetch_by_asin(asin):
    """Look up a specific product by its ASIN."""
    results = fetch_page(asin, page=1)
    # Try exact match first
    for r in results:
        if str(r.get('asin', '')).upper() == asin.upper():
            return r
    # Fallback to first result
    if results:
        print(f'  ⚠ Exact ASIN not found, using closest match.')
        return results[0]
    raise RuntimeError(f'ASIN {asin} returned no results.')

def fetch_top_competitors(query, target_asin, kws, n=5):
    results = fetch_page(query, page=1)
    return [r for r in results
            if matches_category(r, kws) and r.get('asin') != target_asin][:n]

In [76]:
print(f'\nTarget ASIN: {TARGET_ASIN}')
print(f'Competitor query: "{COMPETITOR_QUERY}"')
print('=' * 70)

print(f'\n▸ Fetching target product by ASIN…')
target = fetch_by_asin(TARGET_ASIN)
target_asin = target.get('asin', TARGET_ASIN)
print(f'  ✓ {str(target.get("title",""))[:70]}')

# Infer category from target title
cat_kws = infer_category(target.get('title', ''))
print(f'\nCategory filter: {cat_kws[:6] if cat_kws else "NONE"}')

print(f'\n▸ Fetching top competitors (page 1)…')
comps = fetch_top_competitors(COMPETITOR_QUERY, target_asin, cat_kws, n=TOP_N_COMP)
print(f'  {len(comps)} competitors found.\n')

print('▸ Scoring all products…')
rows = [score_product(target)] + [score_product(c) for c in comps]
df = pd.DataFrame(rows)
df.index = ['TARGET'] + [f'COMP-{i+1}' for i in range(len(comps))]

print()
display(df[['title','price','discount','rating','reviews','lqs','grade','ctr','cvr','rpi']])

target_page = 'direct (ASIN)'   


Target ASIN: B0CHX1W1XY
Competitor query: "laptop under 50000"

▸ Fetching target product by ASIN…


RuntimeError: ASIN B0CHX1W1XY returned no results.

In [ ]:
colors = ['#e74c3c'] + ['#3498db'] * len(comps)
labels = list(df.index)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(f'Competitor Analysis — "{SEARCH_QUERY}"\n'
             f'Target buried on page {target_page} (red) vs page 1 leaders (blue)',
             fontsize=13, fontweight='bold')

# RPI bar
ax = axes[0,0]
bars = ax.bar(labels, df['rpi'], color=colors, edgecolor='white')
ax.set_title('Revenue Per Impression (RPI = Price × CTR × CVR)')
ax.set_ylabel('RPI'); ax.set_xticklabels(labels, rotation=20, ha='right')
for b, v in zip(bars, df['rpi']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}',
            ha='center', fontsize=8, fontweight='bold')

# LQS bar
ax = axes[0,1]
gc = {'A':'#27ae60','B':'#f39c12','C':'#e74c3c'}
bars = ax.bar(labels, df['lqs'], color=[gc.get(g,'#95a5a6') for g in df['grade']],
              edgecolor='white')
ax.set_title('Listing Quality Score (LQS)'); ax.set_ylim(60, 100)
ax.set_xticklabels(labels, rotation=20, ha='right')
for b, v, g in zip(bars, df['lqs'], df['grade']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f} ({g})',
            ha='center', fontsize=8, fontweight='bold')

# CTR vs CVR bubble
ax = axes[1,0]
for i,(lbl,row) in enumerate(df.iterrows()):
    size = max(row['reviews']**0.4, 30) * 8
    ax.scatter(row['ctr'], row['cvr'], s=size, c=colors[i],
               alpha=0.75, edgecolors='white', linewidth=1.2, zorder=3)
    ax.annotate(lbl, (row['ctr'], row['cvr']),
                textcoords='offset points', xytext=(5,5), fontsize=8)
ax.set_xlabel('CTR'); ax.set_ylabel('CVR')
ax.set_title('CTR vs CVR (bubble = review count)'); ax.grid(True, alpha=0.3)

# Price vs RPI
ax = axes[1,1]
for i,(lbl,row) in enumerate(df.iterrows()):
    ax.scatter(row['price'], row['rpi'], c=colors[i], s=100,
               alpha=0.8, edgecolors='white', linewidth=1.2, zorder=3)
    ax.annotate(lbl, (row['price'], row['rpi']),
                textcoords='offset points', xytext=(5,5), fontsize=8)
ax.set_xlabel('Price (₹)'); ax.set_ylabel('RPI')
ax.set_title('Price vs RPI'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('competitor_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
t    = df.loc['TARGET']
best = df.drop('TARGET').loc[df.drop('TARGET')['rpi'].idxmax()]
peer_avg = df.drop('TARGET')[['lqs','ctr','cvr','rpi','rating','reviews','price']].mean()

print('=' * 70)
print(f'TARGET    (page {target_page}): {t["title"]}')
print(f'           LQS={t["lqs"]:.1f} ({t["grade"]})  RPI={t["rpi"]:.2f}')
print(f'BEST COMP (page 1)    : {best["title"]}')
print(f'           LQS={best["lqs"]:.1f} ({best["grade"]})  RPI={best["rpi"]:.2f}')
print('=' * 70)

print('\nPERFORMANCE GAP (vs best competitor)')
print(f'  {"Metric":<10} {"Target":>10} {"Best":>10} {"Gap":>10}')
print('  ' + '-'*42)
for m in ['lqs','ctr','cvr','rpi','rating','reviews']:
    d = best[m] - t[m]
    tag = 'worse' if d > 0 else ('better' if d < 0 else 'same')
    print(f'  {m.upper():<10} {t[m]:>10.3g} {best[m]:>10.3g} {d:>+10.3g}  ({tag})')

In [ ]:
print('RPI SIMULATION — what if we improve the listing?')
print(f'  {"Scenario":<25} {"CTR":>7} {"CVR":>7} {"RPI":>10}  vs best')
print('  ' + '-'*56)
for label, c_mul, v_mul in [
    ('Current',       1.00, 1.00),
    ('CTR +10 %',     1.10, 1.00),
    ('CVR +10 %',     1.00, 1.10),
    ('Both +10 %',    1.10, 1.10),
    ('CTR +25 %',     1.25, 1.00),
    ('Both +25 %',    1.25, 1.25),
]:
    ctr_s = min(t['ctr'] * c_mul, 1.0)
    cvr_s = min(t['cvr'] * v_mul, 1.0)
    rpi_s = t['price'] * ctr_s * cvr_s
    delta = ((rpi_s - best['rpi']) / best['rpi'] * 100) if best['rpi'] else 0
    print(f'  {label:<25} {ctr_s:>7.3f} {cvr_s:>7.3f} {rpi_s:>10.2f}  ({delta:+.1f} %)')

for label, ctr_s, cvr_s in [
    ('Match best CTR',       best['ctr'], t['cvr']),
    ('Match best CTR + CVR', best['ctr'], best['cvr']),
]:
    rpi_s = t['price'] * ctr_s * cvr_s
    delta = ((rpi_s - best['rpi']) / best['rpi'] * 100) if best['rpi'] else 0
    print(f'  {label:<25} {ctr_s:>7.3f} {cvr_s:>7.3f} {rpi_s:>10.2f}  ({delta:+.1f} %)')

In [77]:
# ════════════════════════════════════════════════════════════════════
# ACTIONABLE SELLER FEEDBACK
# Diagnoses listing health, prioritizes fixes, gives expected impact.
# ════════════════════════════════════════════════════════════════════

def generate_seller_feedback(target_row, best_row, peer_avg, target_page):
    """Return a structured feedback report for the seller."""
    feedback = {
        'health'     : [],   # what's wrong
        'quick_wins' : [],   # do this week
        'medium_term': [],   # do this month
        'long_term'  : [],   # do this quarter
        'strengths'  : [],   # what's already working
        'priority_score': {} # impact ranking
    }

    t, b = target_row, best_row

    # ── HEALTH DIAGNOSIS ──────────────────────────────────────────────
    if t['lqs'] < 70:
        feedback['health'].append(f'🔴 CRITICAL: LQS={t["lqs"]} (Grade {t["grade"]}) — listing barely meets quality threshold.')
    elif t['lqs'] < 80:
        feedback['health'].append(f'🟡 WARNING: LQS={t["lqs"]} (Grade {t["grade"]}) — room to improve content quality.')
    else:
        feedback['health'].append(f'🟢 HEALTHY: LQS={t["lqs"]} (Grade {t["grade"]}) — strong content baseline.')

    if t['rating'] < 3.5:
        feedback['health'].append(f'🔴 CRITICAL: Rating {t["rating"]} ★ — well below buyer trust threshold (4.0 ★).')
    elif t['rating'] < 4.0:
        feedback['health'].append(f'🟡 WARNING: Rating {t["rating"]} ★ — below Amazon trust floor (4.0 ★).')

    if t['reviews'] < 50:
        feedback['health'].append(f'🔴 CRITICAL: Only {int(t["reviews"])} reviews — buyers see this as risky.')
    elif t['reviews'] < 200:
        feedback['health'].append(f'🟡 WARNING: {int(t["reviews"])} reviews — needs to cross 200 to look established.')

    if t['rpi'] < b['rpi'] * 0.5:
        feedback['health'].append(f'🔴 CRITICAL: RPI {t["rpi"]:.1f} is less than half of leader ({b["rpi"]:.1f}).')

    # ── STRENGTHS ─────────────────────────────────────────────────────
    if t['discount'] > 20:
        feedback['strengths'].append(f'✓ Strong discount ({t["discount"]:.1f}%) — keeps you visible on deals.')
    if t['price'] < peer_avg['price'] * 0.9:
        feedback['strengths'].append(f'✓ Price (₹{t["price"]:.0f}) is below peer average (₹{peer_avg["price"]:.0f}) — value proposition.')
    if t['rating'] >= 4.5:
        feedback['strengths'].append(f'✓ Excellent rating ({t["rating"]} ★) — leverage this in title/A+.')
    if t['ctr'] > b['ctr']:
        feedback['strengths'].append(f'✓ CTR ({t["ctr"]}) actually beats leader ({b["ctr"]}) — clicks aren\'t the problem.')
    if t['cvr'] > b['cvr']:
        feedback['strengths'].append(f'✓ CVR ({t["cvr"]}) beats leader ({b["cvr"]}) — conversion isn\'t the problem.')
    if not feedback['strengths']:
        feedback['strengths'].append('  (No major strengths identified — focus on quick wins below.)')

    # ── QUICK WINS (this week, free/cheap) ────────────────────────────
    if t['discount'] < 10:
        lift = 10  # +10 CTR points
        feedback['quick_wins'].append((
            'Apply a coupon/discount of 10 %+',
            f'Visible discount badges lift CTR by ~{lift} pts. Expected RPI: ₹{t["price"] * min(t["ctr"]+lift/100,1) * t["cvr"]:.1f}'
        ))
    if t['rating'] < 4.0 and t['reviews'] >= 10:
        feedback['quick_wins'].append((
            'Request Review programme + remove old 1-2★ reviews via "report abuse"',
            'Raising rating from current → 4.0 typically lifts CTR & CVR by 5-15 % combined.'
        ))
    if t['ctr'] < 0.4:
        feedback['quick_wins'].append((
            'Refresh main image (white BG, fills 85 % of frame, shows key spec)',
            'Hero image is the #1 CTR driver. Test 2-3 variants via Manage Your Experiments.'
        ))
    if peer_avg['reviews'] > t['reviews'] * 3:
        feedback['quick_wins'].append((
            'Enroll in Amazon Vine to harvest 10-30 verified reviews',
            'Vine reviews from established reviewers carry more weight than ordinary reviews.'
        ))

    # ── MEDIUM TERM (this month) ──────────────────────────────────────
    if t['lqs'] < 80:
        feedback['medium_term'].append((
            'Rewrite title: brand + model + 3 key specs (8-12 words, 100-150 chars)',
            'Title is 4 of the 23 LQS features. Expected LQS lift: +3-5 points.'
        ))
        feedback['medium_term'].append((
            'Add all 5 bullet points with benefit-led copy (not specs)',
            'Each bullet should answer "what\'s in it for me?". Expected LQS lift: +2-3 points.'
        ))
        feedback['medium_term'].append((
            'Upload 7-9 images: lifestyle, infographic, scale shot, packaging',
            'Listings with 7+ images convert 18 % better on average.'
        ))
    if t['cvr'] < b['cvr']:
        feedback['medium_term'].append((
            'Enable Prime (FBA or Seller-Fulfilled Prime)',
            f'Prime badge adds ~15 % CVR. Expected RPI: ₹{t["price"] * t["ctr"] * min(t["cvr"]+0.15, 1):.1f}'
        ))

    # ── LONG TERM (this quarter) ──────────────────────────────────────
    if t['lqs'] < 85:
        feedback['long_term'].append((
            'Build A+ Content / Enhanced Brand Content (5 modules minimum)',
            'A+ lifts CVR by 10-15 %. Requires Brand Registry.'
        ))
        feedback['long_term'].append((
            'Add product video (45-60 sec, unboxing + key feature demo)',
            'Video lifts CVR by 20-30 %. Becomes a strong CVR moat.'
        ))
    if t['reviews'] < 500:
        feedback['long_term'].append((
            f'Drive to 500+ reviews via post-purchase Request Review automation',
            'Crosses the social-proof threshold where reviews stop being a CVR drag.'
        ))
    if t['ctr'] < b['ctr'] * 0.7:
        feedback['long_term'].append((
            'Sponsored Products + Sponsored Brand campaigns at top-of-search',
            'Buys page-1 visibility while organic LQS catches up. Budget ~5-10 % of revenue.'
        ))

    # ── PRIORITY SCORING (impact × ease) ──────────────────────────────
    actions = []
    base_rpi = t['price'] * t['ctr'] * t['cvr']

    # Each action: (label, est_ctr_lift, est_cvr_lift, ease_1to5)
    candidates = [
        ('Improve main image',          0.05,  0.00, 5),
        ('Apply 10%+ discount',         0.10,  0.10, 5),
        ('Rewrite title (5 bullets too)',0.03,  0.03, 4),
        ('Get to 4.0★ rating',          0.08,  0.10, 3),
        ('Reach 200+ reviews',          0.05,  0.08, 2),
        ('Enable Prime',                0.10,  0.15, 3),
        ('Add A+ content',              0.00,  0.13, 3),
        ('Add product video',           0.00,  0.20, 2),
    ]
    for label, ctr_lift, cvr_lift, ease in candidates:
        new_ctr = min(t['ctr'] + ctr_lift, 1.0)
        new_cvr = min(t['cvr'] + cvr_lift, 1.0)
        new_rpi = t['price'] * new_ctr * new_cvr
        impact = new_rpi - base_rpi
        # Priority score: higher impact + higher ease (ease scaled 1-5)
        priority = impact * ease / 5
        actions.append({
            'action'    : label,
            'rpi_lift'  : round(impact, 1),
            'ease'      : ease,
            'priority'  : round(priority, 1),
        })
    actions.sort(key=lambda x: x['priority'], reverse=True)
    feedback['priority_score'] = actions

    return feedback


# ── RUN + DISPLAY ─────────────────────────────────────────────────────
fb = generate_seller_feedback(t, best, peer_avg, target_page)

print('═' * 72)
print('  SELLER FEEDBACK REPORT')
print('═' * 72)
print(f'\n  Product : {t["title"]}')
print(f'  ASIN    : {t["asin"]}')
print(f'  Position: Page {target_page}  |  LQS = {t["lqs"]} ({t["grade"]})  |  RPI = ₹{t["rpi"]}')

print('\n┌─ HEALTH CHECK ' + '─'*56)
for line in fb['health']:
    print(f'│ {line}')

print('\n┌─ STRENGTHS (keep these) ' + '─'*47)
for line in fb['strengths']:
    print(f'│ {line}')

print('\n┌─ QUICK WINS — DO THIS WEEK ' + '─'*43)
for i, (action, why) in enumerate(fb['quick_wins'], 1):
    print(f'│ {i}. {action}')
    print(f'│    → {why}')

print('\n┌─ MEDIUM TERM — DO THIS MONTH ' + '─'*41)
for i, (action, why) in enumerate(fb['medium_term'], 1):
    print(f'│ {i}. {action}')
    print(f'│    → {why}')

print('\n┌─ LONG TERM — DO THIS QUARTER ' + '─'*41)
for i, (action, why) in enumerate(fb['long_term'], 1):
    print(f'│ {i}. {action}')
    print(f'│    → {why}')

print('\n┌─ PRIORITY ACTION RANKING (impact × ease) ' + '─'*30)
print(f'│ {"Rank":<5} {"Action":<35} {"RPI Lift":>10} {"Ease":>6} {"Priority":>10}')
print(f'│ {"-"*5} {"-"*35} {"-"*10} {"-"*6} {"-"*10}')
for i, a in enumerate(fb['priority_score'], 1):
    print(f'│ #{i:<4} {a["action"]:<35} ₹{a["rpi_lift"]:>8} {a["ease"]:>5}/5 {a["priority"]:>10}')

print('\n┌─ EXECUTIVE SUMMARY ' + '─'*52)
print(f'│ Status      : Page {target_page} — {("BURIED" if target_page > 2 else "NEAR-PAGE-1")}')
print(f'│ Gap to top  : RPI ₹{best["rpi"] - t["rpi"]:.1f} short of best competitor (₹{best["rpi"]:.1f})')
print(f'│ Top action  : {fb["priority_score"][0]["action"]}')
print(f'│ Est. lift   : +₹{fb["priority_score"][0]["rpi_lift"]} RPI from #1 action alone')
print(f'│ To reach #1 : Apply top 3 actions in sequence (est. {sum(a["rpi_lift"] for a in fb["priority_score"][:3]):.0f} ₹ combined lift)')
print('└' + '─'*70)
print('\n✓ Feedback report complete.')

════════════════════════════════════════════════════════════════════════
  SELLER FEEDBACK REPORT
════════════════════════════════════════════════════════════════════════

  Product : Dell Inspiron 5440 Notebook Computer, Intel Core 3 100U 14th Gen 
  ASIN    : B0GNK6BHB2
  Position: Page 4  |  LQS = 76.3 (B)  |  RPI = ₹8697.84

┌─ HEALTH CHECK ────────────────────────────────────────────────────────
│ 🟡 WARNING: LQS=76.3 (Grade B) — room to improve content quality.
│ 🔴 CRITICAL: Only 2 reviews — buyers see this as risky.
│ 🔴 CRITICAL: RPI 8697.8 is less than half of leader (21145.8).

┌─ STRENGTHS (keep these) ───────────────────────────────────────────────
│ ✓ Strong discount (51.7%) — keeps you visible on deals.

┌─ QUICK WINS — DO THIS WEEK ───────────────────────────────────────────
│ 1. Enroll in Amazon Vine to harvest 10-30 verified reviews
│    → Vine reviews from established reviewers carry more weight than ordinary reviews.

┌─ MEDIUM TERM — DO THIS MONTH ────────────────────

In [78]:
print(type(model))                  # → xgboost.XGBRegressor
print(model.feature_names_in_)      # → 23 feature names you defined
print(MODEL_PATH)                   # → your local pkl file

<class 'xgboost.sklearn.XGBRegressor'>
['title_length' 'title_word_count' 'title_brand_present'
 'title_sentence_case' 'title_spellcheck_pass' 'bullet_count'
 'bullets_spell_pass' 'bullets_caps_pass' 'final_bullet_flag'
 'description_word_count' 'final_desc_flag' 'price' 'mrp' 'discount_pct'
 'star_rating' 'review_count' 'log_review_count' 'best_seller_rank'
 'image_count' 'video_count' 'has_video' 'has_aplus' 'aplus_check_pass']
C:\Users\tarun\OneDrive\Desktop\Opsell\LQS_Project\models\lqs_model.pkl
